# Inductive Bias in Machine Learning - Starter Notebook
Explores inductive bias: the assumptions a learner uses to generalise beyond training data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor

In [ ]:
# Inductive bias taxonomy
bias_types = {
    'Restriction bias': 'Limits the hypothesis space (e.g., linear only)',
    'Preference bias': 'Prefers simpler hypotheses (e.g., Occams razor, MDL)',
    'Procedural bias': 'Induced by search procedure (e.g., ID3 prefers short trees)',
}
for k, v in bias_types.items():
    print(f'{k}:\n  {v}\n')

algorithm_biases = {
    'Find-S':        'Restriction: only conjunctive hypotheses; Preference: maximally specific',
    'Decision Tree': 'Preference: shorter trees (Occam), tests near root get high IG',
    'Linear Model':  'Restriction: linear decision boundary only',
    'kNN':           'Preference: smooth decision boundaries (locality)',
    'Naive Bayes':   'Restriction: feature independence assumption',
}
print('Algorithm -> Inductive Bias:')
for alg, bias in algorithm_biases.items():
    print(f'  {alg:<18}: {bias}')

In [ ]:
# Empirical demo: restriction bias — linear vs polynomial vs decision tree
np.random.seed(42)
X_train = np.sort(np.random.uniform(0, 1, 12))[:, None]
y_train = np.sin(2 * np.pi * X_train.ravel()) + np.random.normal(0, 0.15, 12)
X_plot = np.linspace(0, 1, 200)[:, None]

models = {
    'Linear (degree=1)':    make_pipeline(PolynomialFeatures(1), LinearRegression()),
    'Polynomial (degree=4)': make_pipeline(PolynomialFeatures(4), LinearRegression()),
    'Polynomial (degree=9)': make_pipeline(PolynomialFeatures(9), LinearRegression()),
    'Decision Tree (depth=1)': DecisionTreeRegressor(max_depth=1),
    'Decision Tree (depth=8)': DecisionTreeRegressor(max_depth=8),
}

fig, axes = plt.subplots(1, len(models), figsize=(18, 3), sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    ax.scatter(X_train, y_train, color='black', s=30, zorder=3)
    ax.plot(X_plot, model.predict(X_plot), color='steelblue')
    ax.set_title(name, fontsize=8)
    ax.set_xticks([])

fig.suptitle('Inductive Bias: Different models fit the same data differently', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# No Free Lunch theorem illustration
# Without inductive bias no learner can outperform random on unseen data across all problems
from sklearn.metrics import mean_squared_error
X_test = np.sort(np.random.uniform(0, 1, 50))[:, None]
y_test = np.sin(2 * np.pi * X_test.ravel())

print('Test MSE (true function = sin(2πx)):')
for name, model in models.items():
    mse = mean_squared_error(y_test, model.predict(X_test))
    print(f'  {name:<30}: MSE={mse:.4f}')